# Sales forecasting
## No Waste Allocation Tool
### What will be our unit forecast of a given product in next 3 months, by Store ?

------------------------------

## Package import

In [1]:
# Importing libraries
import os
import numpy as np
import pandas as pd, datetime
import seaborn as sns
import matplotlib.pyplot as plt
from time import time
from ydata_profiling import ProfileReport 
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Ridge
from math import sqrt
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

#### Loading & Preprocessing Data

In [2]:
path = r"C:\Users\johnfredrick.owotoru\OneDrive - L'Oréal\Desktop\No Waste Pro\Data\Outlet H00 FRA sellout data.csv"

In [3]:
# Importing store data
store_data= pd.read_csv(path)

# Set display options to show numbers as whole numbers
pd.set_option('display.float_format', '{:.0f}'.format)
store_data.head()

,store_code,store_name,day,sap_code,barcode,source_code,group_reference,item_type,mars_article_id,label,...,signature,franchise,brand,subbrand,axis,subaxis,class,subclass,sales_incl,total_items
0,H00,H00 OUTLET THE VILLAGE,2024-06-19,AA043900,3600523827756,3600523827756,P230084R,MAR,3600523827756 X,CR LipLiner Cout Nu 107 Seine Sunse,...,L'Oreal Paris,L'Oreal Make-Up,L'Oreal Cosmetics,L'Oreal Color Riche,MakeUp,Lip Makeup,Lip Liner,Lip Liners,6,1
1,H00,H00 OUTLET THE VILLAGE,2024-06-19,AA043901,3600523827756,AA043900,P230084R,MAR,AA043900 X,RAL CR LIP LINER 107 SEINE SUNS.NU,...,L'Oreal Paris,L'Oreal Make-Up,L'Oreal Cosmetics,L'Oreal Color Riche,MakeUp,Lip Makeup,Lip Liner,Lip Liners,6,1
2,H00,H00 OUTLET THE VILLAGE,2024-06-19,AA043901,3600523827756,AA043901,P230084R,MAR,AA043901 X,CR LIPLINER COUT NU 107 SEINE SUNSE,...,L'Oreal Paris,L'Oreal Make-Up,L'Oreal Cosmetics,L'Oreal Color Riche,MakeUp,Lip Makeup,Lip Liner,Lip Liners,6,1
3,H00,H00 OUTLET THE VILLAGE,2024-06-19,AA043901,3600523827756,AA043901,P230084R,MAR,AA043901 X,CR LIPLINER COUT NU 107 SEINE SUNSE,...,L'Oreal Paris,L'Oreal Make-Up,L'Oreal Cosmetics,L'Oreal Color Riche,MakeUp,Lip Makeup,Lip Liner,Lip Liners,6,1
4,H00,H00 OUTLET THE VILLAGE,2024-06-19,AA043902,3600523827756,3600523827756,P230084R,MAR,3600523827756 X,CR LIPLINER COUT NU 107 SEINE SUNSE,...,L'Oreal Paris,L'Oreal Make-Up,L'Oreal Cosmetics,L'Oreal Color Riche,MakeUp,Lip Makeup,Lip Liner,Lip Liners,6,1


In [4]:
#Selecting relevant columns
sel_cols = ['store_code', 'day', 'barcode', 'division', 'signature', 'subbrand', 'subaxis', 'subclass', 'sales_incl', 'total_items']
df = store_data[sel_cols]
df.head()

#Print unique values in each column for initial exploration
for k in df.columns:
  print(k, df[k].nunique())
  print('===============================')

store_code 1
day 337
barcode 4395
division 4
signature 32
subbrand 663
subaxis 25
subclass 198
sales_incl 2709
total_items 167


In [5]:
# Display the datatype info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 851013 entries, 0 to 851012
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   store_code   851013 non-null  object 
 1   day          851013 non-null  object 
 2   barcode      847648 non-null  float64
 3   division     847649 non-null  object 
 4   signature    847649 non-null  object 
 5   subbrand     846292 non-null  object 
 6   subaxis      836281 non-null  object 
 7   subclass     831615 non-null  object 
 8   sales_incl   851013 non-null  float64
 9   total_items  851013 non-null  int64  
dtypes: float64(2), int64(1), object(7)
memory usage: 64.9+ MB


In [6]:
# Make a backup copy of the initial store data
store_data = df.copy()

In [7]:
# Convert 'day' column to datetime objects
store_data['day'] = pd.to_datetime(store_data['day'])

In [8]:
# Date Feature Engineering to extract date components
store_data['Year'] = store_data['day'].dt.year
store_data['Month'] = store_data['day'].dt.month
store_data['Day'] = store_data['day'].dt.day
store_data['WeekOfYear'] = store_data['day'].dt.isocalendar().week
store_data['YearMonth'] = store_data['Year'].astype(str) + '-' + store_data['Month'].astype(str)

# Set 'day' as index 
store_data = store_data.set_index('day')
store_data.head(2)

,store_code,barcode,division,signature,subbrand,subaxis,subclass,sales_incl,total_items,Year,Month,Day,WeekOfYear,YearMonth
day,,,,,,,,,,,,,,
2024-06-19,H00,3600523827756,CPD,L'Oreal Paris,L'Oreal Color Riche,Lip Makeup,Lip Liners,6,1,2024,6,19,25,2024-6
2024-06-19,H00,3600523827756,CPD,L'Oreal Paris,L'Oreal Color Riche,Lip Makeup,Lip Liners,6,1,2024,6,19,25,2024-6


#### Exploratory Data Analysis

In [9]:
# EDA using Pandas Prifiling
profile = ProfileReport(store_data, title ="Dataset Summary")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
# More data exploration
print(store_data.shape)
print(store_data.duplicated().sum())

(851013, 14)
690722


In [11]:
# Handle duplicates
store_data.drop_duplicates(keep='first', inplace=True) 

In [12]:
# checking for missing values
store_data.isnull().sum()

store_code         0
barcode          633
division         632
signature        632
subbrand        1351
subaxis        10769
subclass       13351
sales_incl         0
total_items        0
Year               0
Month              0
Day                0
WeekOfYear         0
YearMonth          0
dtype: int64

In [13]:
# Some Data Visualizations
plt.figure(figsize=(12, 6))
sns.lineplot(x='Day', y='total_items', data=store_data)
plt.title('Daily Trend of Total Item sold Over Time')
plt.show()

plt.figure(figsize=(12, 6))
sns.lineplot(x='YearMonth', y='total_items', data=store_data)
plt.title('Year of Month Trend of Total Item sold Over Time')
plt.show()

In [15]:
# Data Type Correction
store_data['WeekOfYear'] = store_data['WeekOfYear'].astype('int64')
store_data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 160291 entries, 2024-06-19 to 2024-06-19
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   store_code   160291 non-null  object 
 1   barcode      159658 non-null  float64
 2   division     159659 non-null  object 
 3   signature    159659 non-null  object 
 4   subbrand     158940 non-null  object 
 5   subaxis      149522 non-null  object 
 6   subclass     146940 non-null  object 
 7   sales_incl   160291 non-null  float64
 8   total_items  160291 non-null  int64  
 9   Year         160291 non-null  int32  
 10  Month        160291 non-null  int32  
 11  Day          160291 non-null  int32  
 12  WeekOfYear   160291 non-null  int64  
 13  YearMonth    160291 non-null  object 
dtypes: float64(2), int32(3), int64(2), object(7)
memory usage: 16.5+ MB


#### Data Cleaning and Preprocessing for Modeling

In [16]:
# Separate target variable and features
target_col = 'total_items'
target = store_data[target_col]
store_data_feat = store_data.drop(columns=[target_col])

In [17]:
# Split numerical and categorical features
num_data = store_data_feat.select_dtypes([float, int])
cat_data = store_data_feat.select_dtypes([object])

##### Correcting missing values for numerical data

In [18]:
# Impute Missing Numerical Data (using mode for barcode)
mode = int(num_data['barcode'].value_counts().idxmax())
num_data['barcode'] = num_data['barcode'].replace(np.nan, mode)
mode

800897064433

#### Correcting missing values for categorical data

In [19]:
cat_data.isnull().sum()

store_code        0
division        632
signature       632
subbrand       1351
subaxis       10769
subclass      13351
YearMonth         0
dtype: int64

In [20]:
# Same method using mode/frequency to replace NaN values, inputing mode result directly
print('division: ', cat_data['division'].value_counts().idxmax())
cat_data['division'] = cat_data['division'].replace(np.nan, 'CPD')

print('signature: ', cat_data['signature'].value_counts().idxmax())
cat_data['signature'] = cat_data['signature'].replace(np.nan, 'NYX Prof. Make-up')

print('subbrand: ', cat_data['subbrand'].value_counts().idxmax())
cat_data['subbrand'] = cat_data['subbrand'].replace(np.nan, 'Nyx Matte')

print('subaxis: ', cat_data['subaxis'].value_counts().idxmax())
cat_data['subaxis'] = cat_data['subaxis'].replace(np.nan, 'Eye Makeup')

print('subclass: ', cat_data['subclass'].value_counts().idxmax())
cat_data['subclass'] = cat_data['subclass'].replace(np.nan, 'Liquid Lipsticks')

division:  CPD
signature:  NYX Prof. Make-up
subbrand:  Nyx Matte
subaxis:  Eye Makeup
subclass:  Liquid Lipsticks


In [21]:
cat_data.isnull().sum()

store_code    0
division      0
signature     0
subbrand      0
subaxis       0
subclass      0
YearMonth     0
dtype: int64

In [22]:
store_data_used = pd.concat([num_data, cat_data, target], axis=1)
store_data_used.head(2)

,barcode,sales_incl,Year,Month,Day,WeekOfYear,store_code,division,signature,subbrand,subaxis,subclass,YearMonth,total_items
day,,,,,,,,,,,,,,
2024-06-19,3600523827756,6,2024,6,19,25,H00,CPD,L'Oreal Paris,L'Oreal Color Riche,Lip Makeup,Lip Liners,2024-6,1
2024-06-19,3600523827787,6,2024,6,19,25,H00,CPD,L'Oreal Paris,L'Oreal Color Riche,Lip Makeup,Lip Liners,2024-6,1


In [23]:
store_data_used.division.value_counts()

division
CPD    148356
LLD      7741
PPD      3560
ACD       634
Name: count, dtype: int64

#### Reclassifying Function for categorical

In [24]:
def signature_reclass(signature):
    if 'NYX Prof. Make-up' in signature:
        return 'NYX_Prof'
    if "L'Oreal Paris" in signature:
        return "LOreal_Paris"
    if 'Maybelline' in signature:
        return 'Maybelline'
    if 'Garnier' in signature:
        return 'Garnier'
    if 'La Provençale Bio' in signature:
        return 'La_Provencale'
    if 'Autres Public' in signature:
        return 'Autres Public'
    if 'Essie' in signature:
        return 'Essie'
    if 'Lancome' in signature:
        return 'Lancome'
    if "L'Oreal Professionnel" in signature:
        return 'LOreal_Professional'
    if 'Yves Saint Laurent':
        return 'Yves_Saint_Laurent'
    else:
        return 'Other signature'    

def subbrand_reclass(sub):
    if "L'Oreal Infaillible" in sub:
        return 'LOreal_Infaillible'
    if "Nyx High Definition" in sub:
        return 'NYX_High_Definition'
    if 'MNY Eye Studio' in sub:
        return 'MNY_Eye_Studio'
    if 'Nyx Suede Matte' in sub:
        return 'NYX_Suede_Matte'
    if 'Nyx Lip Lingerie' in sub:
        return 'NYX_Lip_Lingerie'
    else:
        return 'Other subbrand'
    

def subaxis_reclass(subaxis):
    if 'Lip Makeup' in subaxis:
        return 'Lip_Makeup'
    if 'Eye Makeup' in subaxis:
        return 'Eye_Makeup'
    if 'Face Makeup' in subaxis:
        return 'Face_Makeup'
    if 'Nail Makeup' in subaxis:
        return 'Nail_Makeup'
    if 'Hair Care' in subaxis:
        return 'Hair_Care'
    if 'Face Care' in subaxis:
        return 'Face_Care'
    if 'Other Makeup' in subaxis:
        return 'Other_makeup'
    if 'Face Cleansing' in subaxis:
        return 'Face_Cleansing'
    if 'Body Care' in subaxis:
        return 'Body_Care'
    if 'Sun Care' in subaxis:
        return 'Sun_Care'
    else:
        return 'Other subaxis'
    

def subclass_reclass(subclas):
    if 'Liquid Lipsticks' in subclas:
        return 'Liquid_Lipsticks'
    if 'Concealers' in subclas:
        return 'Concealers'
    if 'Liquid Foundations' in subclas:
        return 'Liquid_Foundations'
    if 'Lip Gloss' in subclas:
        return 'Lip_Gloss'
    if 'Stick lipsticks' in subclas:
        return 'Stick_lipsticks'
    else:
        return 'Other subclass'

In [25]:
# Applying the reclassification function
store_data_used['signature']=store_data_used['signature'].apply(signature_reclass)
store_data_used['subbrand']=store_data_used['subbrand'].apply(subbrand_reclass)
store_data_used['subaxis']=store_data_used['subaxis'].apply(subaxis_reclass)
store_data_used['subclass']=store_data_used['subclass'].apply(subclass_reclass)

In [26]:
#Filter only rows greater or equal to 0
store_data_used = store_data_used[store_data_used['total_items'] >= 0]

In [27]:
store_data_used.columns

Index(['barcode', 'sales_incl', 'Year', 'Month', 'Day', 'WeekOfYear',
       'store_code', 'division', 'signature', 'subbrand', 'subaxis',
       'subclass', 'YearMonth', 'total_items'],
      dtype='object')

In [28]:
# Defining columns of interest and Target variable
cols=['barcode', 'sales_incl', 'Year', 'Month', 'Day', 'store_code', 'division','signature', 'subbrand','subaxis','subclass']
target=store_data_used['total_items']

In [29]:
# selecting feature data of columns of interest into a new DataFrame
feat_data=store_data_used[cols]

In [30]:
# separating numberical and categorical data
num = feat_data.select_dtypes([float, int])
cat = feat_data.select_dtypes([object])

### Preprocessing 

In [31]:
# Defining the Scaling function for the numerical dataset
def num_scaling(num_data):
    scaler = StandardScaler()
    minmax = MinMaxScaler()
    num_scaled = scaler.fit_transform(num_data)
    minmax_scaled = minmax.fit_transform(num_scaled)

    num_prep = pd.DataFrame(minmax_scaled, index=num_data.index, columns=num_data.columns)

    return scaler, minmax, num_prep

In [32]:
# Applying the scaling function
scaler, minmax, num_preprocessed = num_scaling(num_data=num)
num_preprocessed.head()

,barcode,sales_incl,Year,Month,Day
day,,,,,
2024-06-19,0,1,1,0,1
2024-06-19,0,1,1,0,1
2024-06-19,0,1,1,0,1
2024-06-19,0,1,1,0,1
2024-06-19,0,1,1,0,1


In [ ]:
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # Use handle_unknown
cat_encoded = onehot.fit_transform(cat)
cat_encoded_df = pd.DataFrame(cat_encoded, columns=onehot.get_feature_names_out(cat.columns), index=cat.index) # Convert to DataFrame with correct column names

In [ ]:
# Combining the processed numerical and categorical data
df_processed = pd.concat([num_preprocessed, cat_encoded, target],axis=1)
df_processed.head()

#### Model Training & Evaluation

Split Data into training and testing sets

Time series data requires a different splitting technique to avoid discontinuity

In [ ]:
# Convert the date string to a Timestamp
split_end_ts = pd.Timestamp('2024-09-01 00:00:00')

try:
    # Find the numerical index corresponding to the timestamp in the original store_data
    split_index = store_data.reset_index()[store_data.reset_index()['day'] == split_end_ts].index[0]


    # Use the calculated numerical index to split the processed data
    X_train = X[:split_index]
    X_test = X[split_index:]
    y_train = y[:split_index]
    y_test = y[split_index:]

except IndexError: # Handle potential cases where there are now no matching dates due to duplicate removal
    print("No matching date found for splitting after duplicate removal. Using percentage split instead.")
    split_point = int(len(X) * 0.8)  # 80/20 split

    X_train = X[:split_point]
    X_test = X[split_point:]
    y_train = y[:split_point]
    y_test = y[split_point:]

In [ ]:
# Model Evaluation Function
def model_evaluation(y_pred, y_test):
    mse = mean_squared_error(y_test, y_pred)   # Order of arguments corrected
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)  # Order of arguments corrected
    r2 = r2_score(y_test, y_pred)              # Order of arguments corrected
    print(f'Mean Squared Error (MSE): {mse}')
    print(f'Root Mean Squared Error (RMSE): {rmse}')
    print(f'Mean Absolute Error (MAE): {mae}')
    print(f'R-squared (R2): {r2}')
    return mse, rmse, mae, r2  # Return evaluation metrics

#### Decision Tree Regressor

In [ ]:
#Decision Tree Model
seed = 42
dtr = DecisionTreeRegressor(random_state=seed)
dtr

In [ ]:
# Training the model
dtr.fit(X_train, y_train)

In [ ]:
# Making Predictions with Decision Trees
dtr_pred = dtr.predict(X_test)

y_pred_dtr_df = pd.Series(dtr_pred, index=y_test.index)

In [ ]:
#Evaluate Decision Tree

dtr_metrics = model_evaluation(y_pred=y_pred_dtr_df, y_test=y_test)

mean_squared_error is: 19.28058130677195
root mean squared error is 4.39096587401587
mean absolute error is: 2.1682163081902206
coefficient of determination is: -305551.47754546796


#### Random Forest Regressor

In [ ]:
# Random Forest Model
num_estimator = 100
seed = 42
rf_model = RandomForestRegressor(n_estimators=num_estimator, random_state=seed)

In [ ]:
%%time
rf_model.fit(X_train, y_train)

In [ ]:
# Make Predictions with Random Forest
rf_pred = rf_model.predict(X_test)
y_pred_rf_df = pd.Series(rf_pred, index=y_test.index)

RandomForestRegressor(random_state=42)

In [ ]:
# Evaluate Random Forest
rf_metrics = model_evaluation(y_pred=y_pred_rf_df, y_test=y_test)

(60164, 43)

(60164,)


## Performance comparison

In [ ]:
model_comparison = pd.DataFrame({'DecisionTree': dtr_metrics, 'Random Forest': rf_metrics}, index = ['MSE', 'RMSE', 'MAE', 'R2'])
print("\n Model Comparison: \n\n", model_comparison)

###### RandomForestRegressor is our Chosen model - Top performing

### Decision Function to Test Model using supplied parameters

In [ ]:
scaler
minmax
onehot
rf_model


def prediction(barcode, sales_incl, Year, Month, Day, store_code, division, signature, subbrand, subaxis, subclass):
    cols = ['barcode', 'sales_incl', 'Year', 'Month', 'Day', 'store_code', 'division','signature', 'subbrand','subaxis','subclass']
    input_array = np.array([[barcode, sales_incl, Year, Month, Day, store_code, division, signature, subbrand, subaxis, subclass]])
    input_df = pd.DataFrame(input_array, columns=cols)

    
    num = input_df[['barcode', 'sales_incl', 'Year', 'Month', 'Day']] 
    cat = input_df[['store_code', 'division', 'signature', 'subbrand', 'subaxis', 'subclass']] 

    num_sts = scaler.transform(num)
    num_encoded = minmax.transform(num_sts)
    num_df = pd.DataFrame(num_encoded, columns=num.columns)

    cat_encoded = onehot.transform(cat)
    cat_df = cat_encoded

    feature = pd.concat([num_df, cat_df], axis=1)
    predictions = rf_model.predict(feature)
    return round(predictions[0])

##### <font color = 'green'> Testing the model on user input

In [ ]:
#Generate a sample to run a test
sample = store_data.sample()
sample

,store_code,barcode,division,signature,subbrand,subaxis,subclass,sales_incl,total_items,Year,Month,Day,WeekOfYear,YearMonth
day,,,,,,,,,,,,,,
2024-05-14,H00,800897848408,CPD,NYX Prof. Make-up,Nyx Primer,Face Makeup,Makeup Bases,12,2,2024,5,14,20,2024-5


#### Model prediction based on trained features

In [ ]:
prediction(barcode=3600523674602, sales_incl=8, Year=2024, Month=12, Day=7, store_code='H00', division='CPD', signature="LOreal_Paris", subbrand="Other subbrand", subaxis="Other subaxis", subclass="Concealers")

1

##### Further work on model Inference Pipeline...

## <font color = 'green'> FORECASTING THE TOTAL ITEMS 

We would need to develop a predicting model that can also predict the numerical columns - `sales_incl`

In [ ]:
target_col = 'sales_incl'
target = store_data[target_col]

store_data_feat = store_data.drop(columns=[target_col])
num_data = store_data_feat.select_dtypes([float, int])
cat_data = store_data_feat.select_dtypes([object])

# Using mode (most frequent value) to replace all NaN in the barcode column
mode = int(num_data['barcode'].value_counts().idxmax())
num_data['barcode'] = num_data['barcode'].replace(np.nan, mode)

# Same method using mean to replace NaN values, inputing mode result directly
cat_data['division'] = cat_data['division'].replace(np.nan, 'CPD')
cat_data['signature'] = cat_data['signature'].replace(np.nan, 'NYX Prof. Make-up')
cat_data['subbrand'] = cat_data['subbrand'].replace(np.nan, 'Nyx Matte')
cat_data['subaxis'] = cat_data['subaxis'].replace(np.nan, 'Eye Makeup')
cat_data['subclass'] = cat_data['subclass'].replace(np.nan, 'Liquid Lipsticks')

store_data_used = pd.concat([num_data, cat_data, target], axis=1)

# Applying the reclassification function
store_data_used['signature']=store_data_used['signature'].apply(signature_reclass)
store_data_used['subbrand']=store_data_used['subbrand'].apply(subbrand_reclass)
store_data_used['subaxis']=store_data_used['subaxis'].apply(subaxis_reclass)
store_data_used['subclass']=store_data_used['subclass'].apply(subclass_reclass)

#Filter only rows greater or equal to 0
store_data_used = store_data_used[store_data_used['total_items'] >= 0]

# Defining columns of interest and Target variable
cols=['barcode', 'Year', 'Month', 'Day', 'store_code', 'division','signature', 'subbrand','subaxis','subclass', 'total_items']
target=store_data_used['sales_incl']

# selecting feature data of columns of interest into a new DataFrame
feat_data=store_data_used[cols]

# separating numberical and categorical data
num = feat_data.select_dtypes([float, int])
cat = feat_data.select_dtypes([object])

In [ ]:
# Splitting the features (X) and target (y) dataset
X = df_processed.drop(columns=['sales_incl'])  # Correct target column name
y = df_processed['sales_incl']

# Calculate split point (e.g., 80/20 split)
split_point = int(len(X) * 0.8)

# Split using integer indexing
X_train = X[:split_point]
X_test = X[split_point:]
y_train = y[:split_point]
y_test = y[split_point:]

In [ ]:
sales_model = Ridge(random_state=seed)

sales_model.fit(X_train, y_train)

y_pred = sales_model.predict(X_test)

y_pred_df = pd.Series(y_pred, index=y_test.index)

model_evaluation(y_pred=y_pred_df, y_test=y_test)

mean_squared_error is: 1431.7301492444549
root mean squared error is 37.83821017496011
mean absolute error is: 11.069772503708597
coefficient of determination is: -5.5059943416327


##### <font color = 'red'> Build the model that uses time function to predict the total_items that will be used for futuristic `sales_incl` data

In [ ]:
time_cols = ['Year', 'Month', 'Day', 'total_items']
sales_incl_total_item = store_data_used[time_cols]

tgt_col = 'total_items'
feat = sales_incl_total_item.drop(columns=[tgt_col])
tgt = sales_incl_total_item[tgt_col]

scaler_sales_total_item, minmax_sales_total_item, num_preprocessed = num_scaling(num_data=feat)

# Calculate the split point to prevent data leakage. 
split_point = int(len(feat) * 0.8)  # 80% train, 20% test - adjust as needed


X_train = feat[:split_point]
X_test = feat[split_point:]

y_train = tgt[:split_point]
y_test = tgt[split_point:]



sales_total_item_model = Ridge(random_state=seed)

sales_total_item_model.fit(X_train, y_train)

y_pred = sales_total_item_model.predict(X_test)

y_pred_df = pd.Series(y_pred, index=y_test.index)

model_evaluation(y_pred=y_pred_df, y_test=y_test)

mean_squared_error is: 19.28058130677195
root mean squared error is 4.39096587401587
mean absolute error is: 2.1682163081902206
coefficient of determination is: -305551.47754546796


##### <font color = 'red'> Inferencing pipeline for forecasting the future `sales_incl`

In [ ]:
scaler_sales_total_item
minmax_sales_total_item
sales_total_item_model
scaler_sales
minmax_sales
onehot_sales
sales_model

def future_data(start_date, periods, freq, barcode, store_code, division, signature, subbrand, subaxis, subclass):
    future_dates = pd.date_range(start = start_date, periods = periods, freq = freq)
    future_data = pd.DataFrame(
        {
            'day': future_dates,
            'barcode': [barcode] * len(future_dates),
            'store_code': [store_code] * len(future_dates),
            'division': [division] * len(future_dates),
            'signature': [signature] * len(future_dates),
            'subbrand': [subbrand] * len(future_dates),
            'subaxis': [subaxis] * len(future_dates),
            'subclass': [subclass] * len(future_dates)   
        }
    )
    # Date Feature Engineering (all date features)
    future_data['day'] = pd.to_datetime(future_data['day'])
    future_data['Year'] = future_data['day'].dt.year
    future_data['Month'] = future_data['day'].dt.month
    future_data['Day'] = future_data['day'].dt.day
    future_data['WeekOfYear'] = future_data['day'].dt.isocalendar().week
    future_data['YearMonth'] = future_data['Year'].astype(str) + '-' + future_data['Month'].astype(str)
    
    # Set 'day' as index 
    future_data = future_data.set_index('day')

    # Predicting the total_items in the future
    time_cols = ['Year', 'Month', 'Day']
    sales_incl_total_item = future_data[time_cols]
    total_future = scaler_sales_total_item.transform(sales_incl_total_item)
    total_future_scaled = minmax_sales_total_item.transform(total_future)
    num_prep = pd.DataFrame(total_future_scaled, index=sales_incl_total_item.index, columns=sales_incl_total_item.columns)
    total_items_pred = sales_total_item_model.predict(num_prep)
    total_items_pred_df = pd.DataFrame(total_items_pred, columns=['total_items'], index=sales_incl_total_item.index)
    
    # Concatenate with the future data to be able to forecast the future sales_incl
    future_data_for_sales_incl = pd.concat([future_data, total_items_pred_df], axis=1)

    # Selecting the columns for forecasting the future sales_incl
    cols = ['barcode', 'Year', 'Month', 'Day', 'store_code', 'division','signature', 'subbrand','subaxis','subclass', 'total_items']
    # selecting feature data of columns of interest into a new DataFrame
    feat_data = future_data_for_sales_incl[cols]
    
    # separating numerical and categorical data and transforming them
    num = feat_data.select_dtypes([float, int])
    cat = feat_data.select_dtypes([object])
    num_enc = scaler_sales.transform(num)
    num_scaled = minmax_sales.transform(num_enc)
    num_df = pd.DataFrame(num_scaled, columns=num.columns)
    cat_encoded = onehot_sales.transform(cat)
    cat_encoded = cat_encoded.reset_index(drop=True)
    feature = pd.concat([num_df, cat_encoded], axis=1)

    # Predicting the future sales_incl
    future_sales_pred = sales_model.predict(feature)
    future_sales_pred_df = pd.DataFrame(future_sales_pred, columns = ['sales_incl'], index = feat_data.index)
    # Full future values for predicting the total items
    future = pd.concat([feat_data, future_sales_pred_df], axis = 1)
    future = future.drop(columns=['total_items'])
    
    return future

##### <font color = 'red'> Inferencing Pipeline for Predicting the `Total_Items`

In [ ]:
scaler
minmax
onehot
rf_model

def future_total_item_prediction(start_date, periods, freq, barcode, store_code, division, signature, subbrand, subaxis, subclass):
    future_values = future_data(start_date, periods, freq, barcode, store_code, division, signature, subbrand, subaxis, subclass)
    cols = ['barcode', 'sales_incl', 'Year', 'Month', 'Day', 'store_code', 'division','signature', 'subbrand','subaxis','subclass']
    future_df = future_values[cols]
    # separating numerical and categorical data
    num = future_df.select_dtypes([float, int])
    cat = future_df.select_dtypes([object])
    num_enc = scaler.transform(num)
    num_scaled = minmax.transform(num_enc)
    num_df = pd.DataFrame(num_scaled, columns=num.columns)
    cat_encoded = onehot.transform(cat)
    cat_encoded = cat_encoded.reset_index(drop=True)
    features = pd.concat([num_df, cat_encoded], axis=1)
     
    # Predicting the future sales_incl
    total_items_pred = rf_model.predict(features)
    total_items_pred_df = pd.DataFrame(total_items_pred, columns = ['total_items_forecasted'], index = future_df.index)
    # Full future values for predicting the total items
    f_data = future_df.reset_index(drop=True)
    forecast = pd.concat([future_df, total_items_pred_df], axis = 1)
    fore = forecast.drop(columns=["sales_incl"])
    print(f"The total item(s) forecasted for unit {barcode} for {periods} day(s) is : {round(fore['total_items_forecasted'].sum())} item(s)")
        
    return fore

In [ ]:
#Sample to test model
sample = store_data.sample()
sample

,store_code,barcode,division,signature,subbrand,subaxis,subclass,sales_incl,total_items,Year,Month,Day,WeekOfYear,YearMonth
day,,,,,,,,,,,,,,
2023-11-06,H00,800897003975,CPD,NYX Prof. Make-up,Nyx Lip Lingerie,Lip Makeup,Liquid Lipsticks,8,1,2023,11,6,45,2023-11


In [ ]:
print(store_data_used.store_code.value_counts())
print("===============================")
print(store_data_used.division.value_counts())
print("===============================")
print(store_data_used.signature.value_counts())


store_code
H00    160263
Name: count, dtype: int64
division
CPD    148338
LLD      7734
PPD      3557
ACD       634
Name: count, dtype: int64
signature
NYX_Prof               63967
LOreal_Paris           34434
Maybelline             29925
Yves_Saint_Laurent      9249
Garnier                 7313
La_Provencale           5759
Autres Public           3053
Essie                   2884
Lancome                 2131
LOreal_Professional     1548
Name: count, dtype: int64


In [ ]:
print(store_data_used.subbrand.value_counts())
print("===============================")
print(store_data_used.subaxis.value_counts())

subbrand
Other subbrand         142307
LOreal_Infaillible       5307
MNY_Eye_Studio           4853
NYX_High_Definition      3411
NYX_Suede_Matte          2239
NYX_Lip_Lingerie         2146
Name: count, dtype: int64
subaxis
Eye_Makeup        46823
Lip_Makeup        34570
Face_Makeup       32602
Hair_Care         14604
Face_Care          9228
Nail_Makeup        7838
Other subaxis      5372
Other_makeup       3187
Face_Cleansing     2535
Body_Care          2046
Sun_Care           1458
Name: count, dtype: int64


In [ ]:
store_data_used.subclass.value_counts()

subclass
Other subclass        104653
Liquid_Lipsticks       24914
Liquid_Foundations      9026
Stick_lipsticks         7775
Concealers              7386
Lip_Gloss               6509
Name: count, dtype: int64

In [ ]:
start_date = "2025-02-13"
periods = 30
freq = "D"
barcode = 3600524068660
store_code = "H00"
division = "CPD"
signature = "LOreal_Paris"
subbrand = "Other subbrand"
subaxis = "Other subaxis"
subclass = "Concealers"

In [ ]:
future_total_item_prediction(start_date, periods, freq, barcode, store_code, division, signature, subbrand, subaxis, subclass)

The total item(s) forecasted for unit 3600524068660 for 30 day(s) is : 26 item(s)


,barcode,Year,Month,Day,store_code,division,signature,subbrand,subaxis,subclass,total_items_forecasted
day,,,,,,,,,,,
2025-02-13,3600524068660,2025,2,13,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-14,3600524068660,2025,2,14,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-15,3600524068660,2025,2,15,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-16,3600524068660,2025,2,16,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-17,3600524068660,2025,2,17,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-18,3600524068660,2025,2,18,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-19,3600524068660,2025,2,19,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-20,3600524068660,2025,2,20,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
2025-02-21,3600524068660,2025,2,21,H00,CPD,LOreal_Paris,Other subbrand,Other subaxis,Concealers,1
